In [1]:
from pathlib import Path
import json
import pandas as pd

RUNS_ROOT = Path("/home/dgu/fin/01_15_new_qlib/runs")
files = sorted(RUNS_ROOT.glob("*/specs/stage4_summary.json"))
print("found:", len(files))

found: 102


In [2]:
from pathlib import Path
import json
import pandas as pd

RUNS_ROOT = Path("/home/dgu/fin/01_15_new_qlib/runs")
files = sorted(RUNS_ROOT.glob("*/specs/stage4_summary.json"))
print("found:", len(files))

def iter_outer_roots(j: dict):
    """
    stage4_summary 최상단이 {"outer_iter_1": {...}, "outer_iter_2": {...}} 처럼
    여러 outer_iter가 있을 수 있어서 모두 yield.
    단일 dict만 있는 경우에도 대응.
    """
    if not isinstance(j, dict):
        return
    # outer_iter_* 키가 있으면 그걸 전부
    outer_keys = [k for k in j.keys() if isinstance(k, str) and k.startswith("outer_iter_")]
    if outer_keys:
        for k in sorted(outer_keys):
            if isinstance(j.get(k), dict):
                yield k, j[k]
    else:
        # 그냥 root dict인 경우
        yield "root", j

def infer_market(root: dict) -> str:
    b = (root or {}).get("benchmark")
    if isinstance(b, dict):
        ins = b.get("insample")
        outs = b.get("outsample")
        if isinstance(ins, dict) and isinstance(outs, dict):
            if {"information_ratio", "net_return"}.issubset(ins) and {"information_ratio", "net_return"}.issubset(outs):
                return "cn"
    return "us"

rows = []

for p in files:
    run_id = p.parents[1].name  # runs/<run_id>/specs/stage4_summary.json
    try:
        with open(p, "r", encoding="utf-8") as f:
            j = json.load(f)

        for outer_key, root in iter_outer_roots(j):
            base = {
                "run_id": run_id,
                "outer_iter": outer_key,
                "market": infer_market(root),
                "hypothesis_id": root.get("hypothesis_id"),
                "backtest_mode": root.get("backtest_mode"),
                "horizon_days": root.get("horizon_days"),
                "in_sample_period": root.get("in_sample_period"),
                "out_sample_period": root.get("out_sample_period"),
            }

            for combo in root.get("all_combinations", []) or []:
                combo_idx = combo.get("combo_idx")

                # (A) optuna 결과 = combo.outsample.excess_return_with_cost
                ex_optuna = (combo.get("outsample", {}) or {}).get("excess_return_with_cost", None)
                rows.append({
                    **base,
                    "combo_idx": combo_idx,
                    "mode": "optuna",
                    "threshold_q": None,  # fixed만 존재
                    "optimal_thresholds": combo.get("optimal_thresholds", None),  # optuna에서 의미있음(있으면)
                    "outsample_excess_return_with_cost": ex_optuna,  # dict 그대로
                })

                # (B) fixed 결과 = combo.fixed_modes[fixed_qXX].outsample.excess_return_with_cost
                fixed_modes = combo.get("fixed_modes", {}) or {}
                for mode_name, mode_data in fixed_modes.items():
                    ex_fixed = (mode_data.get("outsample", {}) or {}).get("excess_return_with_cost", None)
                    rows.append({
                        **base,
                        "combo_idx": combo_idx,
                        "mode": mode_name,  # fixed_q90, fixed_q95 등
                        "threshold_q": mode_data.get("threshold_q", None),
                        "optimal_thresholds": None,  # fixed에는 보통 없음
                        "outsample_excess_return_with_cost": ex_fixed,  # dict 그대로
                    })

    except Exception as e:
        print("fail:", p, e)

df_raw = pd.DataFrame(rows)
df_raw.head()

found: 102


,run_id,outer_iter,market,hypothesis_id,backtest_mode,horizon_days,in_sample_period,out_sample_period,combo_idx,mode,threshold_q,optimal_thresholds,outsample_excess_return_with_cost
0,20260122_100355,root,us,BH_MR_PanicSelling_3D_v1,qlib,3.0,2023-01-01 ~ 2024-12-31,2025-01-01 ~ 2025-12-31,1,optuna,NaN,"{'volume_surge_zscore': 0.71866277978636, 'liq...",None
1,20260122_100355,root,us,BH_MR_PanicSelling_3D_v1,qlib,3.0,2023-01-01 ~ 2024-12-31,2025-01-01 ~ 2025-12-31,2,optuna,NaN,"{'volume_surge_zscore': 0.8614988717085303, 'l...",None
2,20260122_100355,root,us,BH_MR_PanicSelling_3D_v1,qlib,3.0,2023-01-01 ~ 2024-12-31,2025-01-01 ~ 2025-12-31,3,optuna,NaN,"{'volume_surge_zscore': 0.6531239780871886, 'l...",None
3,20260122_100355,root,us,BH_MR_PanicSelling_3D_v1,qlib,3.0,2023-01-01 ~ 2024-12-31,2025-01-01 ~ 2025-12-31,4,optuna,NaN,"{'volume_surge_zscore': 0.7673745974615892, 'l...",None
4,20260122_100355,root,us,BH_MR_PanicSelling_3D_v1,qlib,3.0,2023-01-01 ~ 2024-12-31,2025-01-01 ~ 2025-12-31,5,optuna,NaN,"{'volume_surge_zscore': 0.5533471026087989, 'l...",None


In [3]:
ex_flat = pd.json_normalize(df_raw["outsample_excess_return_with_cost"])
df = pd.concat([df_raw.drop(columns=["outsample_excess_return_with_cost"]), ex_flat], axis=1)

df = df.sort_values(by='information_ratio', ascending=False)
df

,run_id,outer_iter,market,hypothesis_id,backtest_mode,horizon_days,in_sample_period,out_sample_period,combo_idx,mode,threshold_q,optimal_thresholds,mean,std,annualized_return,information_ratio,max_drawdown,net_return
950,20260201_061954,outer_iter_1,cn,BH_MR_ShortTermDip_3D_v1,qlib_native,3.0,2015-01-01 ~ 2020-12-31,2021-01-01 ~ 2025-12-31,3,fixed_q80,0.80,None,0.000620,0.012330,0.147599,0.775968,-0.198011,0.980118
1753,20260202_125915,outer_iter_5,cn,BH_MR_PanicSelling_3D_v1,qlib_native,3.0,2015-01-01 ~ 2020-12-31,2021-01-01 ~ 2025-12-31,12,optuna,NaN,"{'formula008': 0.55, 'formula012': 0.55, 'form...",0.000689,0.019397,0.163872,0.547626,-0.477056,0.841533
1443,20260202_125915,outer_iter_2,cn,BH_MR_PanicSelling_5D_v1,qlib_native,5.0,2015-01-01 ~ 2020-12-31,2021-01-01 ~ 2025-12-31,2,fixed_q95,0.95,None,0.000614,0.021060,0.146118,0.449729,-0.586463,0.611207
870,20260129_201801,outer_iter_5,cn,BH_SR_Rebound_3D_v1,combined_or,NaN,None,None,combined_or,optuna,NaN,{},0.000553,0.019995,0.131498,0.426295,-0.660160,0.538065
2424,20260203_063015,outer_iter_5,cn,TH_MR_LowVolatilityCompression_12D_v1,qlib_native,12.0,2015-01-01 ~ 2020-12-31,2021-01-01 ~ 2025-12-31,17,fixed_q95,0.95,None,0.000354,0.015488,0.084370,0.353102,-0.642173,0.330308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
851,20260129_151128,outer_iter_1,cn,BH_SR_ShortTermRebound_2D_v1,vectorized,2.0,2015-01-01 ~ 2020-12-31,2021-01-01 ~ 2025-12-31,2,optuna,NaN,"{'formula009': 0.7, 'formula011': 0.8, 'formul...",NaN,NaN,NaN,NaN,NaN,NaN
852,20260129_151128,outer_iter_1,cn,BH_SR_ShortTermRebound_2D_v1,vectorized,2.0,2015-01-01 ~ 2020-12-31,2021-01-01 ~ 2025-12-31,3,optuna,NaN,"{'formula009': 0.85, 'formula011': 0.8, 'formu...",NaN,NaN,NaN,NaN,NaN,NaN
853,20260129_151128,outer_iter_1,cn,BH_SR_ShortTermRebound_2D_v1,vectorized,2.0,2015-01-01 ~ 2020-12-31,2021-01-01 ~ 2025-12-31,4,optuna,NaN,"{'formula009': 0.9, 'formula011': 0.8, 'formul...",NaN,NaN,NaN,NaN,NaN,NaN
854,20260129_151128,outer_iter_1,cn,BH_SR_ShortTermRebound_2D_v1,vectorized,2.0,2015-01-01 ~ 2020-12-31,2021-01-01 ~ 2025-12-31,5,optuna,NaN,"{'formula009': 0.8, 'formula012': 0.55, 'formu...",NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df['market'].value_counts()

market
cn    2330
us      95
Name: count, dtype: int64

In [5]:
df['horizon_days'].value_counts()

horizon_days
3.0     1175
5.0      718
8.0      211
10.0     118
2.0      112
12.0      51
7.0       18
Name: count, dtype: int64

In [7]:
df.query('run_id == "20260203_063015" and combo_idx == 12').groupby(['run_id', 'mode', 'outer_iter', 'combo_idx'])['information_ratio'].max().sort_values(ascending=False).reset_index()

,run_id,mode,outer_iter,combo_idx,information_ratio
0,20260203_063015,optuna,outer_iter_5,12,-0.098531
1,20260203_063015,optuna,outer_iter_2,12,-0.153773
2,20260203_063015,fixed_q95,outer_iter_4,12,-0.202207
3,20260203_063015,fixed_q95,outer_iter_1,12,-0.351938
4,20260203_063015,fixed_q95,outer_iter_5,12,-0.471291
5,20260203_063015,fixed_q90,outer_iter_1,12,-0.510189
6,20260203_063015,fixed_q90,outer_iter_2,12,-0.662565
7,20260203_063015,fixed_q95,outer_iter_2,12,-0.699061
8,20260203_063015,fixed_q90,outer_iter_5,12,-0.741279
9,20260203_063015,fixed_q90,outer_iter_4,12,-0.827570
